# 🧠 MS Lesion Segmentation: Training Notebook
### Stage 1: Infrastructure & Data Ingestion

This notebook is designed to handle training on **Google Colab (T4 GPU)** or your **Local Machine** automatically.

## ⚙️ 1.0 Global Configuration & Hardware Switch
This cell detects where the code is running and sets the correct paths automatically.

In [ ]:
import os
import torch

# --- SMART SWITCH ---
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("🌩️ Environment: Google Colab detected.")
    DRIVE_ROOT = "/content/drive/MyDrive"
    BASE_DIR = "/content"
else:
    print("💻 Environment: Local Machine detected.")
    # UPDATE THIS to your local Google Drive path if using the Sync App
    DRIVE_ROOT = "G:/My Drive" 
    BASE_DIR = "."

# Global Workspace Paths
DATA_DIR = os.path.join(BASE_DIR, "data/raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "data/processed")
MODEL_SAVE_PATH = os.path.join(DRIVE_ROOT, "MS_Project/models")

# Hardware Identification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Hardware: Using {device.type.upper()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'})")

## 🛠 1.1 Environment Setup
Install core medical AI libraries and mount Google Drive if in Colab.

In [ ]:
if IN_COLAB:
    !pip install -q monai[all] SimpleITK nibabel tqdm
    from google.colab import drive
    drive.mount('/content/drive')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

print(f"✅ Workspace ready at: {BASE_DIR}")
print(f"💾 Models will persist to: {MODEL_SAVE_PATH}")

## 📊 1.2 Data Ingestion (Hybrid Strategy)
We pull the ZIP files from your **Google Drive** and extract them to **local storage** for maximum training speed.

In [ ]:
import os
import glob

# Path to where your ZIP files are stored on Drive
DRIVE_DATA_SOURCE = os.path.join(DRIVE_ROOT, "brain_dataset")

categories = ["MSLesSeg", "Mendeley", "Long-MR-MS"]

for category in categories:
    drive_folder = os.path.join(DRIVE_DATA_SOURCE, category)
    local_folder = os.path.join(DATA_DIR, category)
    os.makedirs(local_folder, exist_ok=True)
    
    print(f"\n📦 Processing {category}...")
    zips = glob.glob(os.path.join(drive_folder, "*.zip"))
    
    if not zips:
        print(f"  ⚠️ No ZIP files found in {drive_folder}")
        continue
        
    for zip_path in zips:
        print(f"  ⚡ Extracting {os.path.basename(zip_path)}...")
        !unzip -q -o "{zip_path}" -d "{local_folder}"

print("\n🚀 SUCCESS: Data is ready for processing.")

## 🔍 1.2.1 Directory Mapping (Diagnostic)
Run this cell to see exactly where your FLAIR and Mask files are located inside the subfolders.

In [ ]:
import os

def map_directory_structure(root_dir, max_depth=3):
    print(f"🔍 Mapping structure for: {root_dir}\n")
    for root, dirs, files in os.walk(root_dir):
        depth = root.replace(root_dir, '').count(os.sep)
        if depth <= max_depth:
            indent = '  ' * depth
            print(f"{indent}📂 {os.path.basename(root)}/")
            if files:
                for f in files[:3]:
                    print(f"{indent}  📄 {f}")
                if len(files) > 3:
                    print(f"{indent}  ... ({len(files)-3} more files)")

map_directory_structure(DATA_DIR)

## 📂 1.2.2 Data Discovery & Pairing
This cell pairs MRI scans with their Ground Truth masks for training across all datasets.

In [ ]:
import glob

def create_data_list(raw_dir):
    data_list = []
    
    # 1. Long-MR-MS
    long_path = os.path.join(raw_dir, 'Long-MR-MS')
    if os.path.exists(long_path):
        for p_folder in glob.glob(os.path.join(long_path, 'patient*')):
            gt = glob.glob(os.path.join(p_folder, '*_gt.nii.gz'))
            flairs = glob.glob(os.path.join(p_folder, '*_FLAIRreg.nii.gz'))
            if gt and flairs:
                for f in flairs:
                    data_list.append({'image': f, 'label': gt[0]})

    # 2. Mendeley
    mendeley_path = os.path.join(raw_dir, 'Mendeley')
    if os.path.exists(mendeley_path):
        flair_imgs = [f for f in glob.glob(os.path.join(mendeley_path, '**/*-Flair.nii'), recursive=True) 
                      if "LESIONSEG" not in f.upper()]
        for f in flair_imgs:
            p_id = os.path.basename(f).split('-')[0]
            mask = os.path.join(os.path.dirname(f), f"{p_id}-LesionSeg-Flair.nii")
            if os.path.exists(mask):
                data_list.append({'image': f, 'label': mask})

    # 3. MSLesSeg
    msles_path = os.path.join(raw_dir, 'MSLesSeg')
    if os.path.exists(msles_path):
        all_nii = glob.glob(os.path.join(msles_path, '**/*.nii*'), recursive=True)
        mask_map = {}
        for m in all_nii:
            if "_MASK" in m.upper():
                parts = os.path.basename(m).split('_')
                if len(parts) >= 2: mask_map[f"{parts[0]}_{parts[1]}"] = m
        
        flairs = [f for f in all_nii if "FLAIR" in f.upper() and "_MASK" not in f.upper()]
        for f in flairs:
            parts = os.path.basename(f).split('_')
            if len(parts) >= 2:
                key = f"{parts[0]}_{parts[1]}"
                if key in mask_map: data_list.append({'image': f, 'label': mask_map[key]})
    
    print(f"✅ Total Aligned Pairs: {len(data_list)}")
    return data_list

train_files = create_data_list(DATA_DIR)

## 🚀 1.3 GPU-Accelerated Preprocessing
Standards volumes to 1mm isotropic spacing using MONAI GPU transforms.

In [ ]:
import SimpleITK as sitk
import numpy as np
from monai.transforms import (Compose, LoadImage, EnsureChannelFirst, Orientation, Spacing, ScaleIntensity)

def get_gpu_preprocessor(is_mask=False):
    return Compose([
        LoadImage(image_only=True),
        EnsureChannelFirst(),
        Orientation(axcodes="RAS"),
        Spacing(pixdim=(1.0, 1.0, 1.0), mode="bilinear" if not is_mask else "nearest"),
        ScaleIntensity() if not is_mask else Compose([]),
    ])

def apply_n4_cpu(img_path):
    img = sitk.ReadImage(img_path)
    mask = sitk.OtsuThreshold(img, 0, 1)
    corrector = sitk.N4BiasFieldCorrectionImageFilter()
    corrector.SetMaximumNumberOfIterations([20, 20, 20])
    return corrector.Execute(img, mask)

print("✅ Preprocessor logic loaded.")

## ⚡ 1.3.1 Bulk Parallel GPU Preprocessing
Standardizes all patient pairs into the `/processed` folder.

In [ ]:
from tqdm.notebook import tqdm
import nibabel as nib

def process_patient_gpu(idx_pair):
    idx, pair = idx_pair
    subj_dir = os.path.join(PROCESSED_DIR, f"sub-{idx:03d}")
    os.makedirs(subj_dir, exist_ok=True)
    img_out, lbl_out = os.path.join(subj_dir, "image.nii.gz"), os.path.join(subj_dir, "label.nii.gz")
    
    if not os.path.exists(img_out):
        try:
            img_n4 = apply_n4_cpu(pair['image'])
            sitk.WriteImage(img_n4, "temp.nii.gz")
            pre_img = get_gpu_preprocessor(is_mask=False)("temp.nii.gz").to(device)
            pre_lbl = get_gpu_preprocessor(is_mask=True)(pair['label']).to(device)
            nib.save(nib.Nifti1Image(pre_img.cpu().numpy().squeeze(), np.eye(4)), img_out)
            nib.save(nib.Nifti1Image(pre_lbl.cpu().numpy().squeeze(), np.eye(4)), lbl_out)
        except Exception as e: return f"❌ {subj_dir}: {e}"
    return {'image': img_out, 'label': lbl_out}

print(f"🚀 Preprocessing {len(train_files)} patients...")
processed_train_files = []
for item in tqdm(enumerate(train_files), total=len(train_files)):
    res = process_patient_gpu(item)
    if isinstance(res, dict): processed_train_files.append(res)

print(f"\n✅ Done: {len(processed_train_files)} pairs standardized.")

## 🧪 1.4 Architecture: 3D Residual U-Net

In [ ]:
from monai.networks.nets import UNet

def get_model(in_channels=1, out_channels=1):
    return UNet(
        spatial_dims=3, in_channels=in_channels, out_channels=out_channels,
        channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2),
        num_res_units=2, norm="INSTANCE", act="LEAKYRELU", dropout=0.1
    )

print("✅ Architecture loaded.")

## 📉 1.5 Loss & Metrics

In [ ]:
from monai.losses import DiceLoss
import torch.nn as nn
from scipy.ndimage import label

class MSLesionLoss(nn.Module):
    def __init__(self, alpha=1.0, weight=10.0):
        super(MSLesionLoss, self).__init__()
        self.alpha, self.dice_loss = alpha, DiceLoss(sigmoid=True, squared_pred=True)
        self.bce_loss = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([weight]).to(device))
    def forward(self, input, target):
        return self.dice_loss(input, target) + self.alpha * self.bce_loss(input, target)

print("✅ Loss logic ready.")

## 🚀 1.6 The Hardware-Agnostic Training Engine

In [ ]:
from monai.data import PersistentDataset, DataLoader, pad_list_data_collate
from monai.transforms import (Compose, LoadImaged, EnsureChannelFirstd, SpatialPadd, RandCropByPosNegLabeld, ToTensord)
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

def train_model(data_list, epochs=50):
    is_gpu = device.type == "cuda"
    model, loss_fn = get_model().to(device), MSLesionLoss().to(device)
    optimizer = AdamW(model.parameters(), lr=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda') if is_gpu else None
    
    transforms = Compose([
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        SpatialPadd(keys=["image", "label"], spatial_size=(96, 96, 96)),
        RandCropByPosNegLabeld(keys=["image", "label"], label_key="label", spatial_size=(96,96,96), pos=1, neg=1, num_samples=4),
        ToTensord(keys=["image", "label"])
    ])
    
    ds = PersistentDataset(data=data_list, transform=transforms, cache_dir="/content/p_cache")
    loader = DataLoader(ds, batch_size=1, shuffle=True, collate_fn=pad_list_data_collate)
    
    print(f"🚀 Training on {device.type.upper()}...")
    start_time_total = time.time()
    for epoch in range(epochs):
        start_time_epoch, epoch_loss, total_correct, total_pixels = time.time(), 0, 0, 0
        model.train()
        for batch in tqdm(loader, desc=f"Epoch {epoch+1}"):
            images, labels = batch["image"].to(device), batch["label"].to(device)
            optimizer.zero_grad()
            if is_gpu:
                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = loss_fn(outputs, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(images)
                loss = loss_fn(outputs, labels)
                loss.backward()
                optimizer.step()
            epoch_loss += loss.item()
            preds = (outputs.sigmoid() > 0.5).float()
            total_correct += (preds == labels).sum().item()
            total_pixels += labels.numel()
        
        scheduler.step()
        accuracy = total_correct / total_pixels
        print(f"🔥 Epoch {epoch+1} | Loss: {epoch_loss/len(loader):.4f} | Acc: {accuracy:.4%} | Time: {(time.time()-start_time_epoch)/60:.2f}m")
        if (epoch + 1) % 5 == 0: torch.save(model.state_dict(), os.path.join(MODEL_SAVE_PATH, "best_model.pth"))

print("✅ Ready.")
# train_model(processed_train_files)